In [12]:
# Libraries and folder paths
# Only the columns that hold place names are read from each dataset, to keep it fast.

import pandas as pd
import os

pd.set_option('display.max_rows', None)

base = r'C:\Users\LENOVO\OneDrive\OneDrive - National College of Ireland\Machine_learning_Project\dataSet'

processed = base + r'\processed_dataset'
clean     = base + r'\clean_dataset'

ber_county_file = processed + r'\housing_energy_dataset\Final_datset_county.csv'
planning_file   = processed + r'\planning_dataset\planning_2021_onwards.csv'
ppr_file        = processed + r'\property_price_dataset\ppr_2021_onwards.csv'
rent_file       = processed + r'\rent_index_dataset\rent_2021_onwards.csv'

In [13]:
# The 26 counties of the Republic of Ireland
# This is the official list every dataset should end up with.

counties = ['Carlow', 'Cavan', 'Clare', 'Cork', 'Donegal', 'Dublin', 'Galway', 'Kerry',
            'Kildare', 'Kilkenny', 'Laois', 'Leitrim', 'Limerick', 'Longford', 'Louth', 'Mayo',
            'Meath', 'Monaghan', 'Offaly', 'Roscommon', 'Sligo', 'Tipperary', 'Waterford',
            'Westmeath', 'Wexford', 'Wicklow']

print(len(counties), 'counties')

26 counties


In [14]:
# Collecting the raw place names from each dataset
# Looking at what each source actually contains before writing any mapping rule.

# BER: the county lookup table made during normalisation
ber_names = pd.read_csv(ber_county_file)['CountyName'].unique()

# Planning: the planning authority (council) names
planning_names = pd.read_csv(planning_file, usecols=['PlanningAuthority'])['PlanningAuthority'].unique()

# PPR: county names
ppr_names = pd.read_csv(ppr_file, usecols=['county'])['county'].unique()

# Rent: location names (towns, areas, Dublin districts)
rent_names = pd.read_csv(rent_file, usecols=['location'])['location'].unique()

print('BER names:     ', len(ber_names))
print('Planning names:', len(planning_names))
print('PPR names:     ', len(ppr_names))
print('Rent names:    ', len(rent_names))

BER names:      52
Planning names: 31
PPR names:      26
Rent names:     409


In [15]:
print('BER:', list(ber_names), '\n')
print('Planning:', list(planning_names))

BER: ['Co. Carlow', 'Co. Cavan', 'Co. Clare', 'Co. Cork', 'Co. Donegal', 'Co. Dublin', 'Co. Galway', 'Co. Kerry', 'Co. Kildare', 'Co. Kilkenny', 'Co. Laois', 'Co. Leitrim', 'Co. Limerick', 'Co. Longford', 'Co. Louth', 'Co. Mayo', 'Co. Meath', 'Co. Monaghan', 'Co. Offaly', 'Co. Roscommon', 'Co. Sligo', 'Co. Tipperary', 'Co. Waterford', 'Co. Westmeath', 'Co. Wexford', 'Co. Wicklow', 'Cork City', 'Dublin 1', 'Dublin 10', 'Dublin 11', 'Dublin 12', 'Dublin 13', 'Dublin 14', 'Dublin 15', 'Dublin 16', 'Dublin 17', 'Dublin 18', 'Dublin 2', 'Dublin 20', 'Dublin 22', 'Dublin 24', 'Dublin 3', 'Dublin 4', 'Dublin 5', 'Dublin 6', 'Dublin 6W', 'Dublin 7', 'Dublin 8', 'Dublin 9', 'Galway City', 'Limerick City', 'Waterford City'] 

Planning: ['Carlow County Council', 'Cavan County Council', 'Clare County Council', 'Cork City Council', 'Cork County Council', 'Donegal County Council', 'Dublin City Council', 'Dun Laoghaire Rathdown County Council', 'Fingal County Council', 'Galway County Council', 'Galwa

In [16]:
# One rule that works for every source
# Looking at the names above, one simple rule covers all of them:
# 1. If there is a comma (Tullow, Carlow), keep the part after the comma
# 2. Remove Co.  (Co. Cork -> Cork)
# 3. Keep the first word (Cork City -> Cork, Dublin 6 -> Dublin, Cork County Council -> Cork)
# 4. Dublin has four councils. Three of them don't start with "Dublin" (Dun Laoghaire Rathdown, Fingal, South Dublin), so these are set to Dublin by hand.

def to_county(name):
    name = str(name).strip()

    # 1. keep the part after the comma, e.g. 'Ballsbridge, Dublin 4' -> 'Dublin 4'
    if ',' in name:
        name = name.split(',')[1].strip()

    # 2. remove 'Co. ', e.g. 'Co. Cork' -> 'Cork'
    name = name.replace('Co. ', '')

    # 3. keep the first word, e.g. 'Dublin 4' -> 'Dublin'
    first_word = name.split(' ')[0]

    # 4. the Dublin councils whose names don't start with 'Dublin'
    if first_word in ['Dun', 'Fingal', 'South']:
        return 'Dublin'

    return first_word


# quick test on a few tricky names
for test in ['Co. Cork', 'Cork City', 'Dublin 6W', 'Fingal County Council',
             'Waterford City and County Council', 'Ballsbridge, Dublin 4', 'Tullow, Carlow']:
    print(test, '->', to_county(test))

Co. Cork -> Cork
Cork City -> Cork
Dublin 6W -> Dublin
Fingal County Council -> Dublin
Waterford City and County Council -> Waterford
Ballsbridge, Dublin 4 -> Dublin
Tullow, Carlow -> Carlow


In [17]:
# Building the lookup table
# One row for every raw name, with the source it came from and its county.

rows = []

for name in ber_names:
    rows.append({'source': 'ber', 'raw_name': name, 'county': to_county(name)})

for name in planning_names:
    rows.append({'source': 'planning', 'raw_name': name, 'county': to_county(name)})

for name in ppr_names:
    rows.append({'source': 'ppr', 'raw_name': name, 'county': to_county(name)})

for name in rent_names:
    rows.append({'source': 'rent', 'raw_name': name, 'county': to_county(name)})

county_lookup = pd.DataFrame(rows)

print(county_lookup.shape)
county_lookup.head(10)

(518, 3)


,source,raw_name,county
0,ber,Co. Carlow,Carlow
1,ber,Co. Cavan,Cavan
2,ber,Co. Clare,Clare
3,ber,Co. Cork,Cork
4,ber,Co. Donegal,Donegal
5,ber,Co. Dublin,Dublin
6,ber,Co. Galway,Galway
7,ber,Co. Kerry,Kerry
8,ber,Co. Kildare,Kildare
9,ber,Co. Kilkenny,Kilkenny


In [18]:
# Checking every name landed on one of the 26 counties
# If anything went wrong, it would show up here as a name that is not in the official list.

not_matched = county_lookup[~county_lookup['county'].isin(counties)]
print('Names not matched to a county:', len(not_matched))
not_matched

Names not matched to a county: 0


,source,raw_name,county


In [19]:
# Each source should cover all 26 counties
for source in ['ber', 'planning', 'ppr', 'rent']:
    one = county_lookup[county_lookup['source'] == source]
    print(source, '->', one['county'].nunique(), 'counties')

ber -> 26 counties
planning -> 26 counties
ppr -> 26 counties
rent -> 26 counties


In [20]:
# The tricky ones: BER Dublin districts/cities and the planning councils
county_lookup[county_lookup['source'].isin(['ber', 'planning'])]

,source,raw_name,county
0,ber,Co. Carlow,Carlow
1,ber,Co. Cavan,Cavan
2,ber,Co. Clare,Clare
3,ber,Co. Cork,Cork
4,ber,Co. Donegal,Donegal
5,ber,Co. Dublin,Dublin
6,ber,Co. Galway,Galway
7,ber,Co. Kerry,Kerry
8,ber,Co. Kildare,Kildare
9,ber,Co. Kilkenny,Kilkenny


In [21]:
# Adding a county id
# A small numeric id for each county, in alphabetical order.

county_ids = pd.DataFrame({'county_id': range(1, len(counties) + 1),
                           'county': counties})

county_lookup = county_lookup.merge(county_ids, on='county', how='left')

county_ids

,county_id,county
0,1,Carlow
1,2,Cavan
2,3,Clare
3,4,Cork
4,5,Donegal
5,6,Dublin
6,7,Galway
7,8,Kerry
8,9,Kildare
9,10,Kilkenny


In [ ]:
# Saving to clean_dataset
# - county_lookup.csv: every raw name -> county (used by the cleaning notebooks)
# - counties.csv: the 26 counties with their id

os.makedirs(clean, exist_ok=True)

county_lookup.to_csv(clean + r'\county_lookup.csv', index=False)
county_ids.to_csv(clean + r'\counties.csv', index=False)

print('Saved county_lookup.csv', county_lookup.shape)
print('Saved counties.csv', county_ids.shape)

Saved county_lookup.csv (518, 4)
Saved counties.csv (26, 2)


In [23]:
# How the other notebooks will use this
# Load the lookup, keep only the rows for that source, and merge on the raw name. Example for PPR:
# python
# lookup = pd.read_csv(clean + r'\county_lookup.csv')
# ppr_lookup = lookup[lookup['source'] == 'ppr'][['raw_name', 'county', 'county_id']]
#
# ppr = ppr.merge(ppr_lookup, left_on='county', right_on='raw_name', how='left')
#